In [ ]:
%pip install faster-whisper transformers ollama opensmile sounddevice numpy torch torchaudio silero-vad accelerate sentencepiece keyboard

In [ ]:
import queue
import threading
import time
import os
import keyboard

import numpy as np
import sounddevice as sd

from faster_whisper import WhisperModel

import opensmile
import unicodedata

from transformers import pipeline
from silero_vad import (
    load_silero_vad,
    get_speech_timestamps
)
import ollama

In [ ]:
!ollama list

In [ ]:
SAMPLE_RATE = 16000
CHANNELS = 1

LLM_MODEL = "gemma3:1b"

# tempo de cada chunk de áudio
CHUNK_DURATION = 5  # segundos

# pasta temporária
TEMP_DIR = "temp_audio"

os.makedirs(TEMP_DIR, exist_ok=True)

In [ ]:
audio_queue = queue.Queue()

vad_model = load_silero_vad()

In [ ]:
print("Carregando LLM...")

ollama.generate(
    model=LLM_MODEL,
    prompt=".",
    keep_alive=-1
)

print("LLM carregado.")

In [ ]:
print("Carregando Whisper...")

whisper_model = WhisperModel(
    "tiny",
    device="cpu",
    compute_type="int8"
)

print("Whisper carregado.")

In [ ]:
print("Carregando modelo de emoção da voz...")

emotion_classifier = pipeline(
    "audio-classification",
    model="superb/wav2vec2-base-superb-er"
)

print("Modelo de emoção carregado.")

print("Carregando modelo textual...")

text_sentiment = pipeline(
    "text-classification",
    model="tabularisai/multilingual-sentiment-analysis"
)

print("Modelo textual carregado.")


In [ ]:

print("Carregando openSMILE...")

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

print("openSMILE carregado.")

In [ ]:
recorded_chunks = []

last_save_time = time.time()


def audio_callback(indata, frames, time_info, status):

    global recorded_chunks
    global last_save_time

    if status:
        print(status)

    recorded_chunks.append(indata.copy())

    elapsed = time.time() - last_save_time

    if elapsed >= CHUNK_DURATION:

        audio_data = np.concatenate(recorded_chunks, axis=0)

        audio_queue.put(audio_data)

        recorded_chunks = []

        last_save_time = time.time()

# ============================================================
# PROCESSAMENTO
# ============================================================

In [ ]:
teste = ''

In [ ]:
def process_audio():

    while True:

        audio_data = audio_queue.get()

        try:

            audio_np = audio_data.flatten()

            max_val = np.max(np.abs(audio_np))

            if max_val > 0:
                audio_np = audio_np / max_val

            speech_timestamps = get_speech_timestamps(
                audio_np,
                vad_model,
                sampling_rate=SAMPLE_RATE
            )

            if len(speech_timestamps) == 0:
                
                continue

            print("\n===================================================")
            print("NOVO CHUNK PROCESSADO")

            print("\n[TRANSCRIÇÃO]")

            segments, info = whisper_model.transcribe(
                audio_np,
                language="pt"
            )

            full_text = ""

            for segment in segments:
                full_text += segment.text + " "

            full_text = full_text.strip()

            print("Texto:", full_text)

            print("\n[EMOÇÃO VOCAL]")

            result = emotion_classifier(
                audio_np,
                sampling_rate=16000
            )

            top_emotion = max(
                result,
                key=lambda x: x["score"]
            )

            emotion_label = top_emotion["label"].lower()
            emotion_score = top_emotion["score"]

            if emotion_score < 0.4:

                emotion_label = "inconclusiva"

                print("Emoção inconclusiva.")

            else:

                print(f"Emoção principal: {emotion_label}")
                print(f"Confiança: {emotion_score:.2f}")

            features = smile.process_signal(
                audio_np,
                SAMPLE_RATE
            )

            loudness = None
            pitch = None
            mfcc1 = None
            alpha_ratio = None

            if "loudness_sma3_amean" in features.columns:

                loudness = features[
                    "loudness_sma3_amean"
                ].values[0]

            if "F0semitoneFrom27.5Hz_sma3nz_amean" in features.columns:

                pitch = features[
                    "F0semitoneFrom27.5Hz_sma3nz_amean"
                ].values[0]

            if "mfcc1_sma3_amean" in features.columns:

                mfcc1 = features[
                    "mfcc1_sma3_amean"
                ].values[0]

            if "alphaRatioV_sma3nz_amean" in features.columns:

                alpha_ratio = features[
                    "alphaRatioV_sma3nz_amean"
                ].values[0]

            texto_lower = full_text.lower()

            texto_lower = unicodedata.normalize(
                "NFKD",
                texto_lower
            ).encode(
                "ASCII",
                "ignore"
            ).decode(
                "utf-8"
            )

            print("\n[ANÁLISE FINAL IA]")

            fusion_prompt = f"""
Você é um sistema auxiliar de análise emocional para acompanhamento clínico de pacientes.

Sua função é apenas realizar uma fusão técnica e descritiva dos sinais detectados na fala e no conteúdo textual.

REGRAS IMPORTANTES:
- NÃO invente informações.
- NÃO faça diagnósticos médicos.
- NÃO afirme doenças, transtornos ou condições clínicas.
- NÃO conclua que o paciente possui depressão, ansiedade ou qualquer patologia.
- Apenas descreva padrões emocionais observáveis presentes nos dados fornecidos.
- Caso existam indícios emocionais negativos, descreva-os apenas como:
  "possíveis sinais emocionais compatíveis com..."
  ou
  "padrões vocais/textuais associados a..."
- Sempre deixe claro que a análise NÃO substitui avaliação médica profissional.
- Utilize apenas os dados recebidos abaixo.
- Não extrapole além das evidências fornecidas.

OBJETIVO:
Realizar uma fusão entre:
- características emocionais da voz
- emoção vocal detectada
- intensidade e estabilidade vocal
- padrões da fala
- contexto textual da transcrição

para gerar uma análise emocional mais natural, humana e objetiva.

DADOS DA ANÁLISE:

TRANSCRIÇÃO:
{full_text}

EMOÇÃO VOCAL DETECTADA:
{emotion_label}

NÍVEL DE CONFIANÇA:
{emotion_score}

ASPECTOS OBSERVADOS NA VOZ:
- intensidade vocal: {loudness}
- tom e variação da fala: {pitch}
- características do padrão de fala: {mfcc1}
- estabilidade vocal percebida: {alpha_ratio}

INSTRUÇÕES DE RESPOSTA:

Responda SOMENTE nos tópicos abaixo.

1. PADRÕES EMOCIONAIS OBSERVADOS
- descreva emoções percebidas na fala e no texto
- descreva estabilidade emocional percebida
- descreva possíveis sinais emocionais negativos apenas se houver evidências nos dados

2. COERÊNCIA ENTRE VOZ E TEXTO
- avalie se o tom emocional da voz combina com o conteúdo falado
- identifique possíveis inconsistências emocionais

3. ANÁLISE DA VOZ E DA FALA
- descreva ritmo, energia, intensidade e estabilidade vocal
- considere os aspectos observados na voz

4. RESUMO CLÍNICO DESCRITIVO
- gere um resumo curto, humano e técnico
- sem diagnósticos
- sem inferências médicas
- sem conclusões definitivas

Finalize com:
"Esta análise possui caráter exclusivamente auxiliar e não substitui avaliação clínica profissional."
"""

            teste = fusion_prompt

            response = ollama.chat(
                model=LLM_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": "Você é um analisador emocional clínico."
                    },
                    {
                        "role": "user",
                        "content": fusion_prompt
                    }
                ],
                options={
                    "temperature": 0.3,
                    "num_predict": 521q
                },
                think=False
            )

            analysis = response["message"]["content"]

            print(analysis)

        except Exception as e:

            print("Erro:", e)

# ============================================================
# THREAD PROCESSAMENTO
# ============================================================

In [ ]:
processing_thread = threading.Thread(
    target=process_audio,
    daemon=True
)

processing_thread.start()

# ============================================================
# START MICROFONE
# ============================================================

In [ ]:
print("\n===================================================")
print("🎤 Ouvindo microfone em tempo real...")
print("Pressione Q para encerrar")
print("===================================================\n")

try:

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=CHANNELS,
        callback=audio_callback,
    ):

        while True:

            if keyboard.is_pressed("q"):

                print("\nEncerrado.")
                break

            time.sleep(0.1)

except KeyboardInterrupt:

    print("\nEncerrado.")